# Nível 1 — etapa 1: carregamento e limpeza

Este notebook cobre **apenas** a carga de `dados/dados_nivel_1.json`, a normalização monetária com a taxa **lida do JSON** (`taxa_cambio_usd_brl`) e o tratamento defensivo de dados sujos.

A análise de regras de PLD (fracionamento, perfil, etc.) fica para etapas seguintes. Os objetos que esta etapa deixa prontos são `df_bruto`, `df_limpo`, `df_excecoes` e `taxa_cambio_usd_brl`.

## 1. Contrato dos dados e funções de carga

O arquivo não é uma lista plana. A raiz traz:

- `taxa_cambio_usd_brl`: câmbio fixo da prova (esperado **5,4**);
- `operacoes`: lista de eventos.

A taxa **não** é uma constante solta no código: se o JSON mudar, a conversão acompanha o contrato. Moedas diferentes de `BRL`/`USD` e campos obrigatórios ausentes são isolados, não convertidos “no chute”.

In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Any, Final, Literal, TypedDict

import pandas as pd
from IPython.display import display

def resolver_caminho_dados() -> Path:
    nome = Path("dados") / "dados_nivel_1.json"
    bases = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidatos: list[Path] = [
        Path("../dados/dados_nivel_1.json"),
        Path("dados/dados_nivel_1.json"),
    ]
    for base in bases:
        candidatos.append(base / nome)
        candidatos.append(base / "desafio-ia" / nome)
    for caminho in candidatos:
        if caminho.is_file():
            return caminho
    raise FileNotFoundError("dados_nivel_1.json não encontrado a partir do diretório de execução.")


CAMINHO_DADOS: Final[Path] = resolver_caminho_dados()
CAMPOS_OBRIGATORIOS: Final[tuple[str, ...]] = (
    "id",
    "cliente_id",
    "data",
    "valor",
    "moeda",
    "canal",
    "tipo",
    "contraparte",
)
MOEDAS_CONHECIDAS: Final[frozenset[str]] = frozenset({"BRL", "USD"})


class OperacaoBruta(TypedDict, total=False):
    id: str
    cliente_id: str
    data: str | None
    valor: float | int | str | None
    moeda: str
    canal: str
    tipo: str
    contraparte: str
    observacao: str


class PayloadNivel1(TypedDict):
    taxa_cambio_usd_brl: float
    operacoes: list[OperacaoBruta]


@dataclass(frozen=True, slots=True)
class RelatorioQualidade:
    n_arquivo: int
    n_duplicatas_id: int
    ids_duplicados: tuple[str, ...]
    n_sem_data: int
    n_invalida_contrato: int
    n_analitico: int


def carregar_payload(caminho: Path) -> PayloadNivel1:
    """Lê o JSON e valida o envelope mínimo (taxa + lista de operações)."""
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de dados não encontrado: {caminho.resolve()}")

    with caminho.open(encoding="utf-8") as arquivo:
        bruto: Any = json.load(arquivo)

    if not isinstance(bruto, dict):
        raise TypeError("A raiz do JSON deve ser um objeto.")
    if "taxa_cambio_usd_brl" not in bruto:
        raise KeyError("Campo obrigatório ausente: taxa_cambio_usd_brl")
    if "operacoes" not in bruto or not isinstance(bruto["operacoes"], list):
        raise KeyError("Campo obrigatório ausente ou inválido: operacoes")

    taxa = float(bruto["taxa_cambio_usd_brl"])
    if taxa <= 0:
        raise ValueError(f"Taxa de câmbio inválida: {taxa}")

    return {"taxa_cambio_usd_brl": taxa, "operacoes": list(bruto["operacoes"])}


def _texto(valor: Any) -> str:
    if valor is None:
        return ""
    return str(valor).strip()


def _parse_data(valor: Any) -> date | None:
    if valor is None:
        return None
    texto = _texto(valor)
    if texto == "" or texto.lower() in {"nan", "nat", "none", "null"}:
        return None
    convertido = pd.to_datetime(texto, errors="coerce")
    if pd.isna(convertido):
        return None
    return convertido.date()


def _parse_valor(valor: Any) -> float | None:
    if valor is None or _texto(valor) == "":
        return None
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return None
    if numero < 0:
        return None
    return numero


def converter_para_brl(valor: float, moeda: str, taxa_usd_brl: float) -> float:
    codigo = moeda.upper()
    if codigo == "BRL":
        return valor
    if codigo == "USD":
        return valor * taxa_usd_brl
    raise ValueError(f"Moeda não suportada para normalização: {moeda}")


def motivo_quebra_contrato(registro: OperacaoBruta) -> str | None:
    """Devolve o motivo se o registro não puder entrar na base analítica, senão None."""
    faltantes = [campo for campo in CAMPOS_OBRIGATORIOS if campo not in registro]
    if faltantes:
        return f"campos obrigatórios ausentes: {', '.join(faltantes)}"

    if not _texto(registro.get("id")):
        return "id vazio"
    if not _texto(registro.get("cliente_id")):
        return "cliente_id vazio"

    valor = _parse_valor(registro.get("valor"))
    if valor is None:
        return "valor nulo, não numérico ou negativo"

    moeda = _texto(registro.get("moeda")).upper()
    if moeda not in MOEDAS_CONHECIDAS:
        return f"moeda desconhecida: {moeda or '(vazia)'}"

    if _parse_data(registro.get("data")) is None:
        return "data nula, vazia ou não parseável"

    return None


def registros_para_frame(
    operacoes: list[OperacaoBruta],
    taxa_usd_brl: float,
) -> pd.DataFrame:
    linhas: list[dict[str, Any]] = []
    for indice, registro in enumerate(operacoes):
        if not isinstance(registro, dict):
            linhas.append(
                {
                    "indice_arquivo": indice,
                    "id": None,
                    "cliente_id": None,
                    "data": pd.NaT,
                    "valor": None,
                    "moeda": None,
                    "canal": None,
                    "tipo": None,
                    "contraparte": None,
                    "observacao": None,
                    "valor_brl": pd.NA,
                    "motivo_exclusao": "registro não é um objeto JSON",
                }
            )
            continue

        motivo = motivo_quebra_contrato(registro)
        valor = _parse_valor(registro.get("valor"))
        moeda = _texto(registro.get("moeda")).upper()
        data_op = _parse_data(registro.get("data"))
        valor_brl: float | None
        try:
            valor_brl = (
                converter_para_brl(valor, moeda, taxa_usd_brl)
                if valor is not None and moeda in MOEDAS_CONHECIDAS
                else None
            )
        except ValueError as erro:
            valor_brl = None
            motivo = motivo or str(erro)

        linhas.append(
            {
                "indice_arquivo": indice,
                "id": _texto(registro.get("id")) or None,
                "cliente_id": _texto(registro.get("cliente_id")) or None,
                "data": pd.Timestamp(data_op) if data_op is not None else pd.NaT,
                "valor": valor,
                "moeda": moeda or None,
                "canal": _texto(registro.get("canal")).lower() or None,
                "tipo": _texto(registro.get("tipo")).lower() or None,
                "contraparte": _texto(registro.get("contraparte")) or None,
                "observacao": _texto(registro.get("observacao")),
                "valor_brl": valor_brl,
                "motivo_exclusao": motivo,
            }
        )
    return pd.DataFrame(linhas)


print("Funções de carga e validação definidas.")

Funções de carga e validação definidas.


## 2. Leitura do arquivo e conversão USD → BRL

`OP-0013` está em **USD** (remessa internacional, R$ equivalente = `12000 * 5.4 = 64800`). Sem essa normalização, totais e comparações entre clientes misturam unidades e distorcem qualquer regra de valor.

In [2]:
payload: PayloadNivel1 = carregar_payload(CAMINHO_DADOS)
taxa_cambio_usd_brl: float = payload["taxa_cambio_usd_brl"]

df_bruto: pd.DataFrame = registros_para_frame(payload["operacoes"], taxa_cambio_usd_brl)

print(f"Arquivo: {CAMINHO_DADOS.resolve()}")
print(f"Taxa USD/BRL lida do JSON: {taxa_cambio_usd_brl}")
print(f"Operações no arquivo: {len(payload['operacoes'])}")
print()
print("Amostra (incluindo valor original e valor_brl):")
display(df_bruto)
print("Conferência da operação em USD:")
display(df_bruto.loc[df_bruto["moeda"] == "USD", ["id", "valor", "moeda", "valor_brl", "observacao"]])

Arquivo: C:\Users\Luis Felipe\Documents\Bruna-Estudos\desafio-ia\dados\dados_nivel_1.json
Taxa USD/BRL lida do JSON: 5.4
Operações no arquivo: 20

Amostra (incluindo valor original e valor_brl):


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,motivo_exclusao
0,0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0,NaN
1,1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0,NaN
2,2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0,NaN
3,3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,,3300.0,NaN
4,4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,,25900.0,NaN
5,5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,,27000.0,NaN
6,6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,17200.0,NaN
7,7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,15200.0,NaN
8,8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,,16100.0,NaN
9,9,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,17200.0,NaN


Conferência da operação em USD:


,id,valor,moeda,valor_brl,observacao
13,OP-0013,12000.0,USD,64800.0,remessa internacional


## 3. Diagnóstico de qualidade

Antes de corrigir, o diagnóstico deixa explícito o que está errado e quantas linhas são afetadas. Nesta base plantada há dois problemas centrais (e um terceiro, de contrato, que o código também cobre se aparecer).

In [3]:
print("Nulos / NaT por coluna (visão tabular):")
display(df_bruto[["id", "cliente_id", "data", "valor", "moeda", "valor_brl"]].isna().sum().to_frame("nulos"))

ids_duplicados_mask: pd.Series = df_bruto["id"].duplicated(keep=False) & df_bruto["id"].notna()
print(f"Linhas com id duplicado: {int(ids_duplicados_mask.sum())}")
display(df_bruto.loc[ids_duplicados_mask].sort_values(["id", "indice_arquivo"]))

print("Registros com quebra de contrato (motivo_exclusao preenchido):")
display(df_bruto.loc[df_bruto["motivo_exclusao"].notna()])

Nulos / NaT por coluna (visão tabular):


,nulos
id,0
cliente_id,0
data,1
valor,0
moeda,0
valor_brl,0


Linhas com id duplicado: 2


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,motivo_exclusao
6,6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,17200.0,NaN
9,9,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,17200.0,NaN


Registros com quebra de contrato (motivo_exclusao preenchido):


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,motivo_exclusao
17,17,OP-0017,CLI-A-5,NaT,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,4300.0,"data nula, vazia ou não parseável"


### Achado 1 — duplicata exata de `OP-0007`

**O que aparece:** o `id` `OP-0007` ocorre duas vezes, com os mesmos valores em todas as colunas de negócio (cliente `CLI-A-3`, 2026-03-05, R$ 17.200, Pix, contraparte Epsilon Consultoria). São as posições 6 e 9 do array (índices 0-based).

**Interpretação:** não é fracionamento (structuring). Fracionamento nesta base usa **ids distintos** no mesmo dia (`OP-0001`/`OP-0002`/`OP-0003`, por exemplo). Aqui é reprocessamento ou carga duplicada do **mesmo** evento.

**Tratativa:** `drop_duplicates(subset=["id"], keep="first")`. Manter as duas linhas inflaria volume e frequência de `CLI-A-3` e geraria falso positivo em regras de PLD. Não há perda de informação operacional: as linhas são idênticas.

### Achado 2 — `OP-0017` sem data

**O que aparece:** `data` veio `null` no JSON. A observação do próprio registro diz *"data nao capturada pelo sistema"*. Canal `especie`, tipo `deposito`.

**Por que não imputar:** chutar uma data (média da base, data mínima, “hoje”) inventaria a ordem temporal. Regras de fracionamento, janela de 24h/7 dias e perfil do cliente dependem de calendário real. Imputar seria viés, não correção.

**Tratativa:** a operação **sai** da base analítica e permanece em `df_excecoes`, com motivo registrado. O valor e o canal continuam visíveis para revisão operacional, mas **não** entram em métricas que exigem data. Não há campo auxiliar no JSON que permita recuperar a data.

## 4. Aplicação das tratativas

Ordem: (1) isolar quebras de contrato (inclui data nula); (2) remover duplicata de `id` apenas na base que passou no contrato, para não “deduplicar” lixo contra registro válido.

In [4]:
df_excecoes_contrato: pd.DataFrame = df_bruto.loc[df_bruto["motivo_exclusao"].notna()].copy()
df_validos: pd.DataFrame = df_bruto.loc[df_bruto["motivo_exclusao"].isna()].copy()

n_antes_dedup: int = len(df_validos)
df_limpo: pd.DataFrame = (
    df_validos.drop_duplicates(subset=["id"], keep="first")
    .drop(columns=["motivo_exclusao"])
    .sort_values(["cliente_id", "data", "id"], kind="mergesort")
    .reset_index(drop=True)
)
n_duplicatas_id: int = n_antes_dedup - len(df_limpo)

ids_duplicados: tuple[str, ...] = tuple(
    sorted(df_validos.loc[df_validos["id"].duplicated(keep=False), "id"].dropna().unique())
)

relatorio: RelatorioQualidade = RelatorioQualidade(
    n_arquivo=len(df_bruto),
    n_duplicatas_id=n_duplicatas_id,
    ids_duplicados=ids_duplicados,
    n_sem_data=int((df_bruto["data"].isna()).sum()),
    n_invalida_contrato=len(df_excecoes_contrato),
    n_analitico=len(df_limpo),
)

df_excecoes: pd.DataFrame = df_excecoes_contrato.reset_index(drop=True)

print("Relatório de qualidade:")
print(relatorio)
print()
print("Exceções (fora da base analítica):")
display(df_excecoes)
print("Base analítica (df_limpo):")
display(df_limpo)
print("Nulos restantes em df_limpo:")
display(df_limpo.isna().sum().to_frame("nulos"))
print("Clientes distintos:", int(df_limpo["cliente_id"].nunique()))

Relatório de qualidade:
RelatorioQualidade(n_arquivo=20, n_duplicatas_id=1, ids_duplicados=('OP-0007',), n_sem_data=1, n_invalida_contrato=1, n_analitico=18)

Exceções (fora da base analítica):


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,motivo_exclusao
0,17,OP-0017,CLI-A-5,NaT,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,4300.0,"data nula, vazia ou não parseável"


Base analítica (df_limpo):


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl
0,0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,18100.0
1,1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,17300.0
2,2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,,18800.0
3,3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,,3300.0
4,4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,,25900.0
5,5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,,27000.0
6,6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,17200.0
7,7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,15200.0
8,8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,,16100.0
9,10,OP-0010,CLI-A-4,2026-03-03,3800.0,BRL,cartao,pagamento,Alfa Comercio LTDA,,3800.0


Nulos restantes em df_limpo:


,nulos
indice_arquivo,0
id,0
cliente_id,0
data,0
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0


Clientes distintos: 6


## 5. Resultado desta etapa

| Métrica | Valor |
|---|---|
| Operações no JSON | 20 |
| Duplicata de `id` removida | 1 (`OP-0007` repetido) |
| Isolada por data nula | 1 (`OP-0017`) |
| Base analítica | 18 operações, 6 clientes |
| Câmbio | 5,4 (campo `taxa_cambio_usd_brl`) |
| USD normalizado | `OP-0013` → R$ 64.800 |

Objetos para as próximas etapas: `df_limpo` (análise), `df_excecoes` (revisão operacional) e `taxa_cambio_usd_brl`.

## Parte A — agregações e regras determinísticas

A partir de `df_limpo`: volume por cliente, contagem por canal e duas regras em pandas (sem LLM). As colunas `alerta_fracionamento` e `alerta_valor_atipico` entram no próprio DataFrame.

### Agregações

In [5]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id", as_index=False)["valor_brl"]
    .sum()
    .rename(columns={"valor_brl": "volume_brl"})
    .sort_values("volume_brl", ascending=False)
    .reset_index(drop=True)
)

ops_por_canal = (
    df_limpo.groupby("canal", as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
    .sort_values("quantidade", ascending=False)
    .reset_index(drop=True)
)

print("Volume total transacionado por cliente (BRL)")
display(volume_por_cliente)
print("Quantidade de operações por canal")
display(ops_por_canal)

Volume total transacionado por cliente (BRL)


,cliente_id,volume_brl
0,CLI-A-4,79500.0
1,CLI-A-1,57500.0
2,CLI-A-2,52900.0
3,CLI-A-3,48500.0
4,CLI-A-5,12600.0
5,CLI-A-6,10200.0


Quantidade de operações por canal


,canal,quantidade
0,pix,8
1,ted,5
2,boleto,3
3,cartao,2


### Regras determinísticas

**Regra 1 — fracionamento:** marca o *cliente* que, no mesmo dia, fez 3 ou mais operações, a soma passou de R$ 50.000 e nenhuma operação isolada chegou a R$ 20.000. A coluna `alerta_fracionamento` fica `True` em todas as linhas daquele cliente.

**Regra 2 — valor atípico:** marca a *operação* cujo `valor_brl` é maior que 5 vezes a mediana daquele cliente. Só vale para quem tem 4 ou mais operações; os demais ficam `False`.

In [6]:
LIMIAR_SOMA_FRACIONAMENTO = 50_000.0
LIMIAR_OPERACAO_ISOLADA = 20_000.0
FATOR_MEDIANA_ATIPICO = 5
MIN_OPS_ATIPICO = 4

resumo_dia = df_limpo.groupby(["cliente_id", "data"], as_index=False).agg(
    n_ops=("id", "size"),
    soma_brl=("valor_brl", "sum"),
    max_brl=("valor_brl", "max"),
)

# Nenhuma isolada "atinge" 20 mil → máximo estritamente abaixo desse piso.
janela_fracionamento = (
    (resumo_dia["n_ops"] >= 3)
    & (resumo_dia["soma_brl"] > LIMIAR_SOMA_FRACIONAMENTO)
    & (resumo_dia["max_brl"] < LIMIAR_OPERACAO_ISOLADA)
)
clientes_fracionamento = set(resumo_dia.loc[janela_fracionamento, "cliente_id"])

df_limpo["alerta_fracionamento"] = df_limpo["cliente_id"].isin(clientes_fracionamento)

n_ops_cliente = df_limpo.groupby("cliente_id")["id"].transform("size")
mediana_cliente = df_limpo.groupby("cliente_id")["valor_brl"].transform("median")
df_limpo["alerta_valor_atipico"] = (n_ops_cliente >= MIN_OPS_ATIPICO) & (
    df_limpo["valor_brl"] > FATOR_MEDIANA_ATIPICO * mediana_cliente
)

print("Janelas diárias avaliadas na Regra 1")
display(
    resumo_dia.assign(encaixa_regra_1=janela_fracionamento).sort_values(
        ["cliente_id", "data"]
    )
)
print("df_limpo com as duas colunas de alerta")
display(
    df_limpo[
        [
            "id",
            "cliente_id",
            "data",
            "valor_brl",
            "alerta_fracionamento",
            "alerta_valor_atipico",
        ]
    ]
)
print("Clientes com alerta de fracionamento:", sorted(clientes_fracionamento))
print("Operações com valor atípico:")
display(df_limpo.loc[df_limpo["alerta_valor_atipico"]])

Janelas diárias avaliadas na Regra 1


,cliente_id,data,n_ops,soma_brl,max_brl,encaixa_regra_1
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
1,CLI-A-1,2026-03-21,1,3300.0,3300.0,False
2,CLI-A-2,2026-03-14,2,52900.0,27000.0,False
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False
4,CLI-A-4,2026-03-03,1,3800.0,3800.0,False
5,CLI-A-4,2026-03-11,1,5100.0,5100.0,False
6,CLI-A-4,2026-03-18,1,5800.0,5800.0,False
7,CLI-A-4,2026-03-24,1,64800.0,64800.0,False
8,CLI-A-5,2026-03-07,1,2900.0,2900.0,False
9,CLI-A-5,2026-03-16,1,7000.0,7000.0,False


df_limpo com as duas colunas de alerta


,id,cliente_id,data,valor_brl,alerta_fracionamento,alerta_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True,False
3,OP-0004,CLI-A-1,2026-03-21,3300.0,True,False
4,OP-0005,CLI-A-2,2026-03-14,25900.0,False,False
5,OP-0006,CLI-A-2,2026-03-14,27000.0,False,False
6,OP-0007,CLI-A-3,2026-03-05,17200.0,False,False
7,OP-0008,CLI-A-3,2026-03-05,15200.0,False,False
8,OP-0009,CLI-A-3,2026-03-05,16100.0,False,False
9,OP-0010,CLI-A-4,2026-03-03,3800.0,False,False


Clientes com alerta de fracionamento: ['CLI-A-1']
Operações com valor atípico:


,indice_arquivo,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,alerta_fracionamento,alerta_valor_atipico
12,13,OP-0013,CLI-A-4,2026-03-24,12000.0,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,64800.0,False,True


### Validação da Regra 1

O caso que **deve** acender é `CLI-A-1` em 2026-03-09: três envios (R$ 18.100, R$ 17.300, R$ 18.800), soma R$ 54.200 (> 50 mil) e nenhum chega a R$ 20 mil.

O caso **parecido que não entra** é `CLI-A-3` em 2026-03-05: também três Pix no mesmo dia, nenhum isolado atinge R$ 20 mil, mas a soma fica em R$ 48.500 — abaixo do corte. Sem esse contraponto a regra pegaria “qualquer trio no dia”, o que não é o enunciado.

In [7]:
def _janela(cliente: str, dia: str) -> pd.Series:
    recorte = df_limpo.loc[
        (df_limpo["cliente_id"] == cliente) & (df_limpo["data"] == pd.Timestamp(dia))
    ]
    return pd.Series(
        {
            "cliente_id": cliente,
            "data": dia,
            "n_ops": len(recorte),
            "soma_brl": recorte["valor_brl"].sum(),
            "max_brl": recorte["valor_brl"].max(),
            "alerta_no_cliente": bool(recorte["alerta_fracionamento"].iloc[0]),
        }
    )


esperado = _janela("CLI-A-1", "2026-03-09")
parecido = _janela("CLI-A-3", "2026-03-05")
validacao_regra_1 = pd.DataFrame([esperado, parecido])
validacao_regra_1["deveria_alertar"] = [True, False]
validacao_regra_1["regra_acertou"] = (
    validacao_regra_1["alerta_no_cliente"] == validacao_regra_1["deveria_alertar"]
)

display(validacao_regra_1)

assert esperado["alerta_no_cliente"] is True, "CLI-A-1 em 09/03 deveria acionar a Regra 1"
assert parecido["alerta_no_cliente"] is False, "CLI-A-3 em 05/03 não deveria acionar a Regra 1"
print("Validação da Regra 1: captura CLI-A-1 e poupa CLI-A-3.")

,cliente_id,data,n_ops,soma_brl,max_brl,alerta_no_cliente,deveria_alertar,regra_acertou
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True,True,True
1,CLI-A-3,2026-03-05,3,48500.0,17200.0,False,False,True


Validação da Regra 1: captura CLI-A-1 e poupa CLI-A-3.


## Parte B — análise com LLM

Cliente escolhido: **`CLI-A-1`**, único com `alerta_fracionamento`. A LLM recebe o recorte agregado (não o JSON inteiro da base) e deve devolver um JSON validado: `nivel_risco`, `tipologia_suspeita`, `red_flags`, `justificativa`.

A chave fica só no ambiente (`.env` / `GOOGLE_API_KEY` ou `GROQ_API_KEY`). Tentamos Gemini (AI Studio, faixa gratuita) e, se faltar, Groq.

In [8]:
import json
import os
import re
import time
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

for candidato_env in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "desafio-ia" / ".env",
]:
    load_dotenv(candidato_env)

CLIENTE_LLM = "CLI-A-1"
NivelRisco = Literal["baixo", "médio", "alto"]


class AvaliacaoPLD(BaseModel):
    nivel_risco: NivelRisco
    tipologia_suspeita: str = Field(min_length=1)
    red_flags: list[str]
    justificativa: str = Field(min_length=1)

    @field_validator("nivel_risco", mode="before")
    @classmethod
    def _normalizar_risco(cls, valor: Any) -> str:
        texto = str(valor).strip().lower()
        mapa = {
            "baixo": "baixo",
            "baixa": "baixo",
            "low": "baixo",
            "medio": "médio",
            "médio": "médio",
            "media": "médio",
            "média": "médio",
            "medium": "médio",
            "alto": "alto",
            "alta": "alto",
            "high": "alto",
        }
        if texto not in mapa:
            raise ValueError(f"nivel_risco inválido: {valor}")
        return mapa[texto]


ops_cliente = df_limpo.loc[df_limpo["cliente_id"] == CLIENTE_LLM].copy()
contexto_cliente = {
    "cliente_id": CLIENTE_LLM,
    "n_operacoes": int(len(ops_cliente)),
    "volume_brl": float(ops_cliente["valor_brl"].sum()),
    "alerta_fracionamento": bool(ops_cliente["alerta_fracionamento"].iloc[0]),
    "n_valor_atipico": int(ops_cliente["alerta_valor_atipico"].sum()),
    "canais": ops_cliente["canal"].value_counts().to_dict(),
    "operacoes": [
        {
            "id": row.id,
            "data": str(row.data.date()),
            "valor_brl": float(row.valor_brl),
            "moeda": row.moeda,
            "canal": row.canal,
            "tipo": row.tipo,
            "contraparte": row.contraparte,
        }
        for row in ops_cliente.itertuples(index=False)
    ],
}

print("Contexto agregado enviado à LLM")
display(ops_cliente[["id", "data", "valor_brl", "canal", "tipo", "contraparte", "alerta_fracionamento", "alerta_valor_atipico"]])
print(json.dumps(contexto_cliente, ensure_ascii=False, indent=2))

Contexto agregado enviado à LLM


,id,data,valor_brl,canal,tipo,contraparte,alerta_fracionamento,alerta_valor_atipico
0,OP-0001,2026-03-09,18100.0,pix,transferencia_enviada,Alfa Comercio LTDA,True,False
1,OP-0002,2026-03-09,17300.0,pix,transferencia_enviada,Alfa Comercio LTDA,True,False
2,OP-0003,2026-03-09,18800.0,ted,transferencia_enviada,Beta Servicos ME,True,False
3,OP-0004,2026-03-21,3300.0,boleto,pagamento,Gama Distribuidora,True,False


{
  "cliente_id": "CLI-A-1",
  "n_operacoes": 4,
  "volume_brl": 57500.0,
  "alerta_fracionamento": true,
  "n_valor_atipico": 0,
  "canais": {
    "pix": 2,
    "ted": 1,
    "boleto": 1
  },
  "operacoes": [
    {
      "id": "OP-0001",
      "data": "2026-03-09",
      "valor_brl": 18100.0,
      "moeda": "BRL",
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA"
    },
    {
      "id": "OP-0002",
      "data": "2026-03-09",
      "valor_brl": 17300.0,
      "moeda": "BRL",
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA"
    },
    {
      "id": "OP-0003",
      "data": "2026-03-09",
      "valor_brl": 18800.0,
      "moeda": "BRL",
      "canal": "ted",
      "tipo": "transferencia_enviada",
      "contraparte": "Beta Servicos ME"
    },
    {
      "id": "OP-0004",
      "data": "2026-03-21",
      "valor_brl": 3300.0,
      "moeda": "BRL",
      "canal": "boleto",
      "tip

### Dois prompts

**Prompt A (vago):** pede uma opinião genérica, sem papéis, sem schema rígido e sem o critério de fracionamento. Costuma vir mais “achismo” e JSON incompleto.

**Prompt B (PLD):** analista de prevenção à lavagem no Brasil, usa as regras já calculadas, proíbe inventar operação e exige o JSON do contrato. Tendência: `tipologia_suspeita` ligada a *smurfing*/fracionamento e `red_flags` rastreáveis no recorte.

In [9]:
contexto_json = json.dumps(contexto_cliente, ensure_ascii=False)

prompt_vago = f"""Olha esses dados de um cliente e me diz se é suspeito.

{contexto_json}
"""

prompt_pld = f"""Você é analista de PLD/FT em instituição financeira no Brasil.

Avalie SOMENTE o cliente abaixo. Não invente operações, datas ou valores.
O recorte já passou por regras determinísticas: alerta_fracionamento=true significa
3+ operações no mesmo dia, soma > R$ 50.000 e nenhuma isolada >= R$ 20.000.

Contrato de resposta: JSON puro, sem markdown, com as chaves
nivel_risco (baixo|médio|alto), tipologia_suspeita (string),
red_flags (lista de strings objetivas), justificativa (string, cite ids e datas).

Cliente:
{contexto_json}
"""

print("--- Prompt A (vago) ---")
print(prompt_vago)
print("--- Prompt B (PLD) ---")
print(prompt_pld)

--- Prompt A (vago) ---
Olha esses dados de um cliente e me diz se é suspeito.

{"cliente_id": "CLI-A-1", "n_operacoes": 4, "volume_brl": 57500.0, "alerta_fracionamento": true, "n_valor_atipico": 0, "canais": {"pix": 2, "ted": 1, "boleto": 1}, "operacoes": [{"id": "OP-0001", "data": "2026-03-09", "valor_brl": 18100.0, "moeda": "BRL", "canal": "pix", "tipo": "transferencia_enviada", "contraparte": "Alfa Comercio LTDA"}, {"id": "OP-0002", "data": "2026-03-09", "valor_brl": 17300.0, "moeda": "BRL", "canal": "pix", "tipo": "transferencia_enviada", "contraparte": "Alfa Comercio LTDA"}, {"id": "OP-0003", "data": "2026-03-09", "valor_brl": 18800.0, "moeda": "BRL", "canal": "ted", "tipo": "transferencia_enviada", "contraparte": "Beta Servicos ME"}, {"id": "OP-0004", "data": "2026-03-21", "valor_brl": 3300.0, "moeda": "BRL", "canal": "boleto", "tipo": "pagamento", "contraparte": "Gama Distribuidora"}]}

--- Prompt B (PLD) ---
Você é analista de PLD/FT em instituição financeira no Brasil.

Avali

### Chamada, validação e métricas

A resposta passa por Pydantic. Se a LLM devolver texto solto ou JSON quebrado, entra um fallback explícito (`tipologia_suspeita = indeterminada`) em vez de derrubar o notebook.

In [10]:
def _extrair_json(texto: str) -> str:
    limpo = texto.strip()
    if limpo.startswith("```"):
        limpo = re.sub(r"^```(?:json)?\s*", "", limpo)
        limpo = re.sub(r"\s*```$", "", limpo)
    inicio, fim = limpo.find("{"), limpo.rfind("}")
    if inicio >= 0 and fim > inicio:
        return limpo[inicio : fim + 1]
    return limpo


def parse_avaliacao(texto: str) -> tuple[AvaliacaoPLD, bool]:
    try:
        return AvaliacaoPLD.model_validate_json(_extrair_json(texto)), False
    except Exception as erro:
        fallback = AvaliacaoPLD(
            nivel_risco="médio",
            tipologia_suspeita="indeterminada",
            red_flags=["resposta da LLM malformada ou indisponível"],
            justificativa=f"Fallback de validação ({type(erro).__name__}): {texto[:500]}",
        )
        return fallback, True


def chamar_llm(prompt: str) -> tuple[str, dict[str, Any]]:
    google_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
    groq_key = os.getenv("GROQ_API_KEY")
    inicio = time.perf_counter()

    if google_key:
        try:
            from google import genai

            modelo = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")
            cliente = genai.Client(api_key=google_key.strip())
            resposta = cliente.models.generate_content(
                model=modelo,
                contents=prompt,
                config={"response_mime_type": "application/json", "temperature": 0.2},
            )
            uso = getattr(resposta, "usage_metadata", None)
            metricas = {
                "provedor": "gemini",
                "modelo": modelo,
                "latencia_s": round(time.perf_counter() - inicio, 3),
                "tokens_prompt": getattr(uso, "prompt_token_count", None),
                "tokens_resposta": getattr(uso, "candidates_token_count", None),
                "tokens_total": getattr(uso, "total_token_count", None),
            }
            return (resposta.text or ""), metricas
        except Exception as erro_gemini:
            print(f"Gemini indisponível ({type(erro_gemini).__name__}); tentando Groq.")

    if groq_key:
        from groq import Groq

        modelos = [
            os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        ]
        vistos: list[str] = []
        ultimo_erro: Exception | None = None
        cliente_groq = Groq(api_key=groq_key.strip())
        for modelo in modelos:
            if modelo in vistos:
                continue
            vistos.append(modelo)
            try:
                resposta = cliente_groq.chat.completions.create(
                    model=modelo,
                    temperature=0.2,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": "Responda apenas JSON válido."},
                        {"role": "user", "content": prompt},
                    ],
                )
                uso = resposta.usage
                metricas = {
                    "provedor": "groq",
                    "modelo": modelo,
                    "latencia_s": round(time.perf_counter() - inicio, 3),
                    "tokens_prompt": getattr(uso, "prompt_tokens", None),
                    "tokens_resposta": getattr(uso, "completion_tokens", None),
                    "tokens_total": getattr(uso, "total_tokens", None),
                }
                return resposta.choices[0].message.content or "", metricas
            except Exception as erro_groq:
                ultimo_erro = erro_groq
                print(f"Groq modelo {modelo} falhou ({type(erro_groq).__name__}).")
        raise RuntimeError(f"Groq indisponível: {type(ultimo_erro).__name__}")

    metricas = {
        "provedor": None,
        "modelo": None,
        "latencia_s": round(time.perf_counter() - inicio, 3),
        "tokens_prompt": 0,
        "tokens_resposta": 0,
        "tokens_total": 0,
    }
    return (
        '{"nivel_risco":"médio","tipologia_suspeita":"indeterminada",'
        '"red_flags":["GOOGLE_API_KEY ou GROQ_API_KEY ausente"],'
        '"justificativa":"Sem chave de API no ambiente; a chamada não foi disparada."}',
        metricas,
    )


def avaliar_prompt(nome: str, prompt: str) -> dict[str, Any]:
    bruto, metricas = chamar_llm(prompt)
    avaliacao, usou_fallback = parse_avaliacao(bruto)
    return {
        "prompt": nome,
        "avaliacao": avaliacao,
        "usou_fallback": usou_fallback,
        "metricas": metricas,
        "bruto": bruto,
    }


resultado_vago = avaliar_prompt("A — vago", prompt_vago)
resultado_pld = avaliar_prompt("B — PLD", prompt_pld)


def _exibir(pacote: dict[str, Any]) -> None:
    print(f"=== {pacote['prompt']} ===")
    print("Métricas:", pacote["metricas"])
    print("Fallback de parse:", pacote["usou_fallback"])
    display(pacote["avaliacao"].model_dump())
    print("Trecho bruto:")
    print(pacote["bruto"][:1200])
    print()


_exibir(resultado_vago)
_exibir(resultado_pld)

comparacao = pd.DataFrame(
    [
        {
            "prompt": pacote["prompt"],
            "nivel_risco": pacote["avaliacao"].nivel_risco,
            "tipologia_suspeita": pacote["avaliacao"].tipologia_suspeita,
            "n_red_flags": len(pacote["avaliacao"].red_flags),
            "fallback": pacote["usou_fallback"],
            "provedor": pacote["metricas"]["provedor"],
            "latencia_s": pacote["metricas"]["latencia_s"],
            "tokens_total": pacote["metricas"]["tokens_total"],
        }
        for pacote in (resultado_vago, resultado_pld)
    ]
)
print("Comparação lado a lado")
display(comparacao)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini indisponível (ClientError); tentando Groq.


Groq modelo llama-3.1-8b-instant falhou (NotFoundError).


Gemini indisponível (ClientError); tentando Groq.


Groq modelo llama-3.1-8b-instant falhou (NotFoundError).


...

{'nivel_risco': 'médio',
 'tipologia_suspeita': 'indeterminada',
 'red_flags': ['resposta da LLM malformada ou indisponível'],
 'justificativa': 'Fallback de validação (ValidationError): {"suspeito":true,"motivo":"Operações de alto valor em mesmo dia para mesma contraparte via PIX, indicando possível estruturação."}'}

Trecho bruto:
{"suspeito":true,"motivo":"Operações de alto valor em mesmo dia para mesma contraparte via PIX, indicando possível estruturação."}

=== B — PLD ===
Métricas: {'provedor': 'groq', 'modelo': 'openai/gpt-oss-20b', 'latencia_s': 2.371, 'tokens_prompt': 587, 'tokens_resposta': 645, 'tokens_total': 1232}
Fallback de parse: False


{'nivel_risco': 'alto',
 'tipologia_suspeita': 'Transferências em grande volume para mesma contraparte em curto prazo',
 'red_flags': ['3 transferências > R$ 50.000 em 1 dia',
  'Nenhuma operação isolada >= R$ 20.000',
  'Transferências concentradas para Alfa Comercio LTDA',
  'Uso de múltiplos canais (pix, ted) no mesmo dia'],
 'justificativa': 'Operações OP-0001, OP-0002 e OP-0003 realizadas em 2026-03-09 somam R$ 54.200, excedendo R$ 50.000 em um único dia, todas enviadas para Alfa Comercio LTDA via pix e ted, sem nenhuma operação isolada ≥ R$ 20.000, indicando fracionamento.'}

Trecho bruto:
{"nivel_risco":"alto","tipologia_suspeita":"Transferências em grande volume para mesma contraparte em curto prazo","red_flags":["3 transferências > R$ 50.000 em 1 dia","Nenhuma operação isolada >= R$ 20.000","Transferências concentradas para Alfa Comercio LTDA","Uso de múltiplos canais (pix, ted) no mesmo dia"],"justificativa":"Operações OP-0001, OP-0002 e OP-0003 realizadas em 2026-03-09 somam R$ 54.200, excedendo R$ 50.000 em um único dia, todas enviadas para Alfa Comercio LTDA via pix e ted, sem nenhuma operação isolada ≥ R$ 20.000, indicando fracionamento."}

Comparação lado a lado


,prompt,nivel_risco,tipologia_suspeita,n_red_flags,fallback,provedor,latencia_s,tokens_total
0,A — vago,médio,indeterminada,1,True,groq,2.875,849
1,B — PLD,alto,Transferências em grande volume para mesma con...,4,False,groq,2.371,1232


### Leitura da comparação

Sem chave no ambiente, as duas linhas caem no mesmo fallback (isso também testa o parse). Com Gemini/Groq configurado, o esperado é:

- **A** mais genérica (“cliente atípico”, risco inflado ou vago, flags pouco amarradas aos ids).
- **B** amarrada ao fracionamento de `CLI-A-1` em 2026-03-09 (`OP-0001`–`OP-0003`) e, em geral, `nivel_risco` médio/alto com tipologia de smurfing.

Para a entrega com chamada real: copie `.env.example` para `.env`, preencha `GOOGLE_API_KEY` (AI Studio) ou `GROQ_API_KEY`, rode de novo esta parte e salve o notebook.